In [ ]:
import os
import json
import random
from PIL import Image
import logging

import numpy as np
import torch
import transformers
from huggingface_hub import snapshot_download
import sys
sys.path.append('/home/b5bq/pu22650.b5bq/decord/python') # decord没有提供arch64，需要手动编译

/home/b5bq/pu22650.b5bq/miniforge3/envs/internlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [2]:
logging.basicConfig(
    format='[%(levelname)s] %(asctime)s %(message)s',
    level=logging.INFO,
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [3]:

dataDir = r'/scratch/b5bq/pu22650.b5bq/VL-ICL'
dataset = 'open_mi'
n_shot = [0,1,2,4] # at most 4 as there are only 5 support image for openmi
max_new_tokens = 15
seed = 0

In [4]:
MODEL_ID = 'internlm/internlm-xcomposer2d5-7b'
CACHE_DIR = "/scratch/b5bq/pu22650.b5bq/hf_cache" # Use pre-downloaded models
model_path = snapshot_download(repo_id=MODEL_ID, cache_dir=CACHE_DIR, local_files_only=True) # Load the model and tokenizer from the downloaded

In [5]:
def set_random_seed(seed_number):
    # position of setting seeds also matters
    os.environ['PYTHONHASHSEED'] = str(seed_number)
    np.random.seed(seed_number)
    random.seed(seed_number)
    torch.manual_seed(seed_number)
    torch.random.manual_seed(seed_number)
    torch.cuda.manual_seed(seed_number)
    torch.cuda.manual_seed_all(seed_number)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_random_seed(seed)

In [7]:
model = transformers.AutoModel.from_pretrained(model_path, trust_remote_code=True, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, device_map="auto")
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model.tokenizer = tokenizer

/home/b5bq/pu22650.b5bq/miniforge3/envs/internlm/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/b5bq/pu22650.b5bq/miniforge3/envs/internlm/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


Set max length to 16384


/home/b5bq/pu22650.b5bq/miniforge3/envs/internlm/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/b5bq/pu22650.b5bq/miniforge3/envs/internlm/lib/python3.10/site-packages/torch/nn/modules/module.py:2397: UserWarning: for vision_model.embeddings.class_embedding: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/home/b5bq/pu22650.b5bq/miniforge3/envs/internlm/lib/python3.10/site-packages/torch/nn/modules/module.py:2397: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint t

In [8]:
def select_demonstration(n_shot, query=None):
    # use two classes for now
    query_class = query['answer']
    other_class = random.choice([cls for cls in query['classes'] if cls != query_class])
    order_keys = [query_class, other_class] if random.choice([True, False]) else [other_class, query_class]
    answers = {query_class: query_class, other_class: other_class}
    
    n_shot_support = []
    for i in range(n_shot):
        for key in order_keys:
            # For each key, add one shot
            support = {
                'image': [query['support'][key]['images'][i]], 
                'answer': answers[key],
                'question': "This is a"
            }
            n_shot_support.append(support)
    return n_shot_support


In [9]:
def load_image(img_ids, root_path):
    if isinstance(img_ids, str):
        img_ids = [img_ids]
    images = []
    image_paths = []
    for img_id in img_ids:
        image_path = os.path.join(root_path, img_id)
        image = Image.open(image_path).convert('RGB').resize((560, 560))
        images.append(image)
        image_paths.append(image_path)
        
    return images, image_paths

In [10]:
def ICL_I2T_inference(model, tokenizer, query, 
                      n_shot_support, dataDir, max_new_tokens):
    img_id = query['image']
    query_images, query_image_paths = load_image(img_id, dataDir) # only support >= 560, in load_image, resize to 560
    query_text = query['question']
    images = []
    input_text = "\n"
    for i in range(len(n_shot_support)):
        for image_path in n_shot_support[i]['image']:
            image = Image.open(os.path.join(dataDir, image_path)).convert("RGB").resize((560, 560)) # only support >= 560
            image = model.vis_processor(image)
            images.append(image)
            input_text += "<ImageHere>"
        input_text += f"{n_shot_support[i]['question']}\nAnswer: {n_shot_support[i]['answer']}\n"
    for query_image in query_images:
        images.append(model.vis_processor(query_image))
        input_text += "<ImageHere>"
    input_text += f"{query_text}\nAnswer:"
    image = torch.stack(images).to(torch.bfloat16).cuda() # [3, 3, 560, 560]
    image = torch.split(image, 1, dim=0) # to list with each [1, 3, 560, 560]
    predicted_answers, history = model.chat(
        tokenizer, 
        query=input_text, 
        image=image, 
        history=[], 
        do_sample=False, 
        max_new_tokens=max_new_tokens,
        use_cache=False,   # 关键
    )
    return predicted_answers

In [11]:
def eval_questions(dataDir, max_new_tokens, query_meta, model, tokenizer, n_shot):
    results = []

    for query_i, query in enumerate(query_meta):

        n_shot_support = select_demonstration(n_shot, query=query)

        predicted_answer = ICL_I2T_inference(model, tokenizer, query, n_shot_support, dataDir, max_new_tokens)
        query['prediction'] = predicted_answer
        results.append(query)
        logging.info(f'shot:{n_shot}, query:{query_i}, truth:{query["answer"]}, prediction:{predicted_answer}')

    return results

In [12]:

query_file = os.path.join(dataDir, dataset, 'query.json')
print(query_file)

with open(query_file, 'r') as f:
    query_meta = json.load(f)

for shot in n_shot:
    results_dict = eval_questions(dataDir, max_new_tokens, query_meta, model, tokenizer, int(shot))
    # os.makedirs(f"results/{dataset}", exist_ok=True)
    # with open(f"results/{dataset}/internlm-x2_{shot}-shot.json", "w") as f:
    #     json.dump(results_dict, f, indent=4)

# past_length = past_key_values[0][0].shape[2]
# AttributeError: 'NoneType' object has no attribute 'shape'
# 这是因为transformers版本问题，需要锁版本

/scratch/b5bq/pu22650.b5bq/VL-ICL/open_mi/query.json


/home/b5bq/pu22650.b5bq/miniforge3/envs/internlm/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:367: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
[INFO] 2025-12-27 16:40:55 shot:0, query:0, truth:blicket, prediction:SCHOOL BUS
[INFO] 2025-12-27 16:40:56 shot:0, query:1, truth:slation, prediction:False
[INFO] 2025-12-27 16:40:56 shot:0, query:2, truth:shously, prediction:fence
[INFO] 2025-12-27 16:40:56 shot:0, query:3, truth:blicket, prediction:Yes
[INFO] 2025-12-27 16:40:56 shot:0, query:4, truth:perpo, prediction:ferret
[INFO] 2025-12-27 16:40:57 shot:0, query:5, truth:blicket, prediction:Lion
[INFO] 2025-12-27 16:40:57 shot:0, query:6, truth:dax, prediction:DOG
[INFO] 2025-12-27 16:40:57 shot:0, query:7, truth:perpo, prediction:Dog
[INFO] 2025-12-27 16:40:58 shot:0, query:8, truth:perpo, prediction

KeyboardInterrupt: 